# Evaluation Notebook

This file performs evaluation of the diffusion-based predictor, stores results in a Zarr file, and generates plots.

## Configuration

In [ ]:
# Set the default model directory.
model_dir = "/data/group/mas/qmas/results/toy_problem_v1/mappo/toy-problem-ensemble3/wandb/run-20250828_140541-nznhbsh7/files"

# Set the output file.
# output_file = "output.zarr"

# Set the output file automatically.
output_file = "./diffuser_lambda_" + model_dir.split("/")[-2] + ".zarr"

## Setup

In [ ]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main

import os
import yaml
import numpy as np
from copy import deepcopy
from multiprocessing import Process
os.environ["WANDB__SERVICE_WAIT"] = "300"

In [ ]:
def convert_args(argsdict):
    # Convert arguments to the appropriate format.
    arglist = []
    for key, value in argsdict.items():
        if type(value) == dict and "value" in value and not key.startswith("_"):
            v = value["value"]
            if v == None:
                continue
            if type(v) == bool:
                if v:
                    arglist.append(f"--{key}")
                else:
                    arglist.append(f"--no-{key}")
            else:
                arglist.append(f"--{key}")
                arglist.append(str(v))
    return arglist


def wrap_arg_val(val):
    return {"value": val}


def load_args(model_dir):
    # Load arguments from the config file.
    config_file = os.path.join(model_dir, "config.yaml")
    args = yaml.load(open(config_file), Loader=yaml.FullLoader)

    # Set required eval-specific arguments. Do not change these!
    args["use_wandb"] = wrap_arg_val(False)
    args["use_render"] = wrap_arg_val(False)
    args["use_eval"] = wrap_arg_val(True)
    args["model_dir"] = wrap_arg_val(model_dir)
    args["eval_output_file"] = wrap_arg_val(output_file)


    # Set default arguments for evaluation. Feel free to change these!
    args["cuda"] = wrap_arg_val(True)
    args["cuda_idx"] = wrap_arg_val(4)
    args["eval_episodes"] = wrap_arg_val(3)
    args["episode_length"] = wrap_arg_val(300)
    args["seed"] = wrap_arg_val(1)

    return args

# Load arguments for the default model.
args = load_args(model_dir)

# Evaluation

In [ ]:
# Check if the output file already exists. If so, skip this cell.
if os.path.exists(args["eval_output_file"]["value"]):
    print("Output file already exists. Skipping evaluation.")

else:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)
    
    obs_probabilities = np.arange(0.0, 1.0, 0.01).tolist()
    for obs_prob in obs_probabilities:
        argscopy = deepcopy(args)
        argscopy["observation_probability"] = wrap_arg_val(obs_prob)
        argscopy["experiment_name"] = wrap_arg_val(f"diffuser_obs_prob_{obs_prob:.2f}")
        start_process(argscopy)

    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    MAX_PROCESSES = 10
    running_processes = []
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
        running_processes = [p for p in running_processes if p.is_alive()]
    print("All processes finished!")

# Analysis

In [ ]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data for plotting
categories = {
    'tracking_error_mean': {
        'title': 'Tracking Error',
        'ylabel': 'Tracking Error (units distance)',
        'xlabel': 'Observation Probability',
        # Average over all episodes and all timesteps.
        'data': [np.mean(output[experiment]['tracking_error_mean']) for experiment in output]
    },
}
for category in categories:
    values = categories[category]['data']

    # Create bar chart
    # plt.bar(list(output.keys()), values)
    x = np.linspace(0, 1, len(values))
    plt.plot(x, values, marker='o')
    
    title = categories[category]['title']
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Scenario' if "xlabel" not in categories[category] else categories[category]['xlabel'])
    plt.ylabel(f"Timesteps out of {args['episode_length']['value']}" if "ylabel" not in categories[category] else categories[category]['ylabel'])
    plt.title(f"{title}: avg. over {args['eval_episodes']['value']} episodes, seed {args['seed']['value']}")
    plt.show()